# 7.24 同一題，怎樣教不同預算的回答？


現在想讓一個小助理接受 `<low>` 或 `<high>`。題目仍是「先有2件，新增3件，再移除1件，剩幾件？」，正確答案都是 4；我們沒有為 low 發明一個比較不正確的答案。改變的是**提供給模型的條件**，以及訓練時展示的作答方式。

先沿用 [7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html) 的監督式微調（SFT）：把條件與問題當輸入，只有助手示範回答的位置計入代價。下面兩筆是人工示範資料；程式用 [7.1](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html) 的 `render_chat` 把它們排成訓練序列、數出各自的有效目標，不更新參數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import render_chat

records = [
    {"effort": "low", "question": "先有2件，新增3件，再移除1件，剩幾件？", "answer": "4"},
    {"effort": "high", "question": "先有2件，新增3件，再移除1件，剩幾件？", "answer": "2+3=5；5-1=4。答案4。"},
]
for row in records:
    messages = [
        {"role": "user", "content": f"<{row['effort']}> {row['question']}"},
        {"role": "assistant", "content": row["answer"]},
    ]
    x, y = render_chat(messages)
    print(row["effort"], "輸入長度", len(x), "有效目標數", (y != -100).sum().item(), "→", row["answer"])

f 字串把檔位寫成 `<low>`、`<high>`，放在問題前面，所以檔位真的進入模型的輸入。這裡的 `<low>` 是普通文字，依 [6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html) 會按 byte 編碼，不是特殊 token；要讓模型分辨檔位，這樣就夠了。輸出顯示：同一道題，low 的示範只有 2 個有效目標（「4」與 EOS），high 有 27 個。兩筆都只是「一題」，受監督的 token 數卻差了十幾倍。

訓練時，effort 標記一定要真的進入模型。若只在資料表多加一欄、模型根本沒讀到，就沒有教它依條件改變回答。同一個問題的不同示範，可以設計不同的代價遮罩（loss mask，決定哪些位置計代價）；但這裡兩份都要學正確答案與結束，不能只對 high 的過程計分，卻忘了最後答案。

為什麼同一題可以放兩次？模型學的是「給定問題與檔位後，這份輸出應有高機率」。low 和 high 是不同的條件，不是互相衝突的正確答案。可是這組資料最直接教到的是**短答與步驟的形式差異**；它不保證多給計算就更會解題，更沒有重現商業服務未公開的內部推理訓練。即使 high 很會寫長解釋，也要用新題檢查是否更常答對。

資料還有兩條界線要守。第一，同一題的兩個版本必須放在同一份資料裡：不能讓 low 版放訓練資料、high 版放測試資料，再說測試時模型沒見過這題，因為它其實見過同一題的另一個版本。第二，長示範讓模型讀到更多受監督的 token，上面的 2 對 27 就是例子；配對的題數相同，不等於訓練工作量相同。比較時要交代題數、有效目標數與更新步數。

Looped 模型提供另一種做法：同一個正確答案，可以在不同的終止深度（區塊反覆跑到第幾輪就讀出答案）計算代價，內部狀態不必寫成文字。本節只需要知道這條路存在：它改變的是計算路徑，可以和本節用文字條件區分檔位的示範搭配；具體做法見 [14.13–14.14](https://birdhackor.github.io/tiny-perceptron-vlm/14.13.html)。

練習把 high 示範改成「2+3=6；6-1=4」。最後的 4 仍然對，過程卻錯了；這筆資料會教進哪個錯誤？先檢查每個可驗算的步驟，再用 [C.1–C.2](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html) 分開看回答形式與過程是否正確。
